In [1]:
import copy
import gc
import os
import sys
import time
from io import BytesIO
from pathlib import Path

import numpy as np
import open3d as o3d
import torch
import torch as th
import torch.nn as nn
import viser
import viser.transforms as tf
from PIL import Image
from plyfile import PlyData, PlyElement

from gui.utils import merge_meshes
from trellis.pipelines import TrellisTextTo3DPipeline
from trellis.utils import postprocessing_utils

# os.environ["SPCONV_ALGO"] = "native"  # Can be 'native' or 'auto', default is 'auto'.

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[SPARSE] Backend: spconv, Attention: flash_attn


In [17]:
class SpaceControl:
    def __init__(self):
        super().__init__()
        self.pipeline = TrellisTextTo3DPipeline.from_pretrained("gui")
        self.pipeline.cuda()

    def __call__(
        self,
        last_path: str = "gui/meshes/last.ply",
        output_path: str = "sample.glb",
        t0_idx=7,
        text_prompt: str = "shoe",
        image_prompt=None,
        steps=12,
        cfg_strength=7.5,
        rescale=1.0,
    ):
        # normalize last mesh
        last_path = Path(last_path)
        last_normalized_path = last_path.parent / f"{last_path.stem}_normalized{last_path.suffix}"

        mesh = o3d.io.read_triangle_mesh(str(last_path))
        aabb = mesh.get_axis_aligned_bounding_box()
        min_bound = aabb.get_min_bound()
        max_bound = aabb.get_max_bound()
        center = (min_bound + max_bound) / 2
        scale = 1.0 / (max_bound - min_bound).max()
        mesh.translate(-center)
        mesh.scale(scale, center=(0, 0, 0))
        o3d.io.write_triangle_mesh(str(last_normalized_path), mesh)

        # apply Trellis
        outputs = self.pipeline.run(
            text_prompt,
            image_prompt,
            seed=1,
            sparse_structure_sampler_params={
                "steps": steps,
                "cfg_strength": cfg_strength,
                "t0_idx_value": t0_idx,
                "spatial_control_mesh_path": str(last_normalized_path),
            },
        )

        # apply trimesh conversion
        glb = postprocessing_utils.to_glb(
            outputs["gaussian"][0],
            outputs["mesh"][0],
            # Optional parameters
            simplify=0.95,  # Ratio of triangles to remove in the simplification process
            texture_size=1024,  # Size of the texture used for the GLB
        )

        # rotate x-axis clock wise
        rot_matrix = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, -1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0]])

        # denormalize
        glb.apply_scale(1 / scale)
        glb.apply_translation(center)  # bring to original scale and position before saving
        glb.apply_transform(rot_matrix)
        glb.apply_scale(rescale)

        # save as glb
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        glb.export(str(output_path))

        # clean
        gc.collect()
        th.cuda.empty_cache()

In [18]:
space_control = SpaceControl()

Using cache found in /home/rvi/.cache/torch/hub/facebookresearch_dinov2_main


In [19]:
# space_control(output_path=f"results/sample_image1.glb", image_prompt=Image.open("gui/meshes/shoe1.jpg"))
# space_control(output_path=f"results/sample_image1_1.glb", image_prompt=Image.open("gui/meshes/shoe1.jpg"), t0_idx=1)
space_control(
    output_path=f"results/sample_image3_3_scale1.2.glb", rescale=1.2, image_prompt=Image.open("gui/meshes/shoe3.jpg"), t0_idx=3
)
space_control(
    output_path=f"results/sample_image3_3_scale1.3.glb", rescale=1.3, image_prompt=Image.open("gui/meshes/shoe3.jpg"), t0_idx=3
)
space_control(
    output_path=f"results/sample_image3_3_scale1.4.glb", rescale=1.4, image_prompt=Image.open("gui/meshes/shoe3.jpg"), t0_idx=3
)
# space_control(
#     output_path=f"results/sample_image3_2_scale0.9.glb", rescale=0.9, image_prompt=Image.open("gui/meshes/shoe3.jpg"), t0_idx=2
# )
# space_control(
#     output_path=f"results/sample_image3_6_scale0.9.glb", rescale=0.9, image_prompt=Image.open("gui/meshes/shoe3.jpg"), t0_idx=6
# )
# space_control(output_path=f"results/sample_image3_4.glb", image_prompt=Image.open("gui/meshes/shoe3.jpg"), t0_idx=4)

Sampling: 100%|██████████| 25/25 [00:02<00:00, 10.24it/s]


Before postprocess: 175798 vertices, 351588 faces


Decimating Mesh: 100%|██████████[00:01<00:00]


After decimate: 8793 vertices, 17578 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1827.11it/s]


...

Decimating Mesh: 100%|██████████[00:01<00:00]


After decimate: 8792 vertices, 17576 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1836.26it/s]


Found 3805 invisible faces
Dual graph: 26364 edges
Mincut solved, start checking the cut
Removed 2686 faces by mincut
INFO- Loaded 7445 vertices and 14890 faces.

100% done 
After remove invisible faces: 7445 vertices, 14898 faces


Rendering: 100it [00:00, 234.36it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [00:07<00:00, 350.47it/s, loss=0.0238]


In [4]:
# space_control(output_path=f"results/sample_image1.glb", image_prompt=Image.open("gui/meshes/shoe1.jpg"))
# space_control(output_path=f"results/sample_image1_1.glb", image_prompt=Image.open("gui/meshes/shoe1.jpg"), t0_idx=1)
space_control(output_path=f"results/sample_image3_1.glb", image_prompt=Image.open("gui/meshes/shoe3.jpg"), t0_idx=1)
space_control(output_path=f"results/sample_image3_4.glb", image_prompt=Image.open("gui/meshes/shoe3.jpg"), t0_idx=4)

Sampling: 100%|██████████| 25/25 [00:04<00:00,  5.33it/s]


Before postprocess: 227578 vertices, 455168 faces


Decimating Mesh: 100%|██████████[00:01<00:00]


After decimate: 11369 vertices, 22757 faces


/home/rvi/conda/envs/torch/lib/python3.11/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1706.84it/s]
WARNING- Some cuts were necessary to cope with non manifold configuration.


...

Decimating Mesh: 100%|██████████[00:00<00:00]


After decimate: 7980 vertices, 15952 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1895.66it/s]


Found 2766 invisible faces
Dual graph: 23928 edges
Mincut solved, start checking the cut
Removed 6709 faces by mincut
INFO- Loaded 4659 vertices and 9243 faces.

0% done 
After remove invisible faces: 4659 vertices, 9243 faces


Rendering: 100it [00:00, 234.69it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [00:07<00:00, 356.26it/s, loss=0.0276]


In [5]:
space_control(output_path=f"results/sample_image2_1.glb", image_prompt=Image.open("gui/meshes/shoe2.jpg"), t0_idx=1)
space_control(output_path=f"results/sample_image2_7.glb", image_prompt=Image.open("gui/meshes/shoe2.jpg"), t0_idx=7)

Sampling: 100%|██████████| 25/25 [00:03<00:00,  8.01it/s]


Before postprocess: 223246 vertices, 446508 faces


Decimating Mesh: 100%|██████████[00:01<00:00]


After decimate: 11152 vertices, 22324 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1723.15it/s]


...

Decimating Mesh: 100%|██████████[00:00<00:00]


After decimate: 7162 vertices, 14330 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1887.88it/s]


Found 1648 invisible faces
Dual graph: 21493 edges
Mincut solved, start checking the cut
Removed 7028 faces by mincut
INFO- Loaded 3685 vertices and 7302 faces.

100% done 
After remove invisible faces: 3685 vertices, 7303 faces


Rendering: 100it [00:00, 249.24it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [00:07<00:00, 351.63it/s, loss=0.00631]


In [4]:
for i in range(1, 10):
    space_control(output_path=f"results/sample_{i:02d}.glb", image_prompt=Image.open("gui/meshes/shoe2.jpg"), t0_idx=float(i))

Sampling: 100%|██████████| 25/25 [00:04<00:00,  5.37it/s]


Before postprocess: 223324 vertices, 446668 faces


Decimating Mesh: 100%|██████████[00:01<00:00]


After decimate: 11156 vertices, 22332 faces


/home/rvi/conda/envs/torch/lib/python3.11/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1704.69it/s]
WARNING- Some cuts were necessary to cope with non manifold configuration.


...

Decimating Mesh: 100%|██████████[00:00<00:00]


After decimate: 7413 vertices, 14818 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1870.48it/s]


Found 9602 invisible faces
Dual graph: 22227 edges
Mincut solved, start checking the cut
Removed 9602 faces by mincut
INFO- Loaded 2610 vertices and 5216 faces.

0% done 
After remove invisible faces: 2610 vertices, 5216 faces


Rendering: 100it [00:00, 252.38it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [00:06<00:00, 362.10it/s, loss=0.00791]
